# 13. Визуализация распределений входов по слоям модели

В ноутбуке:

- выбирается связка `датасет + модель` из YAML-конфигов проекта;
- подкачивается raw batch изображений из выбранного датасета;
- запускается инференс модели с hook-ами по `nn.Linear`;
- визуализируются распределения активаций (`inputs` или `outputs`) на нескольких слоях.


In [ ]:
import torch

MODEL_YAML_NAME = "clip_vit_b32"
DATASET_YAML_NAME = "coco2017"
DATA_ROOT = "./data_tutorials"
N_IMAGES = 8
DEVICE_OVERRIDE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_KIND = "inputs"  # "inputs" | "outputs"
TOP_K_LAYERS = 8
MAX_POINTS_PER_LAYER = 200_000
HIST_BINS = 60
RANDOM_SEED = 42

raw_dataset = None
model = None

print("MODEL_YAML_NAME:", MODEL_YAML_NAME)
print("DATASET_YAML_NAME:", DATASET_YAML_NAME)
print("DATA_ROOT:", DATA_ROOT)
print("DEVICE_OVERRIDE:", DEVICE_OVERRIDE)
print("TARGET_KIND:", TARGET_KIND)


Если в окружении нет `numpy`/`matplotlib`, установите их и перезапустите kernel:

```bash
%pip install numpy matplotlib
```


In [ ]:
from __future__ import annotations

import math
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from dataset.data_raw.providers.hf import register_all_adapters
from dataset.data_raw.providers.hf.auth import get_hf_token
from dataset.data_raw.registry import create_dataset
from dataset.models.registry import create_model


def load_hydra_cfg(config_path: str = "conf/config.yaml", overrides: list[str] | None = None):
    cfg_path = repo_root / config_path
    with initialize_config_dir(version_base=None, config_dir=str(cfg_path.parent.resolve())):
        return compose(config_name=cfg_path.stem, overrides=overrides or [])


def resolve_dirs(paths: list[str]) -> list[Path]:
    resolved = []
    for item in paths:
        path = Path(str(item))
        if not path.is_absolute():
            path = repo_root / path
        resolved.append(path)
    return resolved


def resolve_dataset_cfg(cfg: Any, dataset_yaml_name: str):
    data_cfg = cfg.data
    config_dirs = list(data_cfg.get("dataset_config_dirs", ["conf/data/datasets"]))
    for cfg_dir in resolve_dirs(config_dirs):
        candidate = cfg_dir / f"{dataset_yaml_name}.yaml"
        if candidate.exists():
            ds_cfg = OmegaConf.load(candidate)
            overrides = data_cfg.get("dataset_overrides", {})
            if overrides and dataset_yaml_name in overrides:
                ds_cfg = OmegaConf.merge(ds_cfg, overrides[dataset_yaml_name])
            return ds_cfg
    raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml_name}")


def resolve_model_cfg(cfg: Any, model_yaml_name: str):
    model_cfg = cfg.models if "models" in cfg else {}
    config_dirs = list(model_cfg.get("model_config_dirs", ["conf/data/models"]))
    for cfg_dir in resolve_dirs(config_dirs):
        candidate = cfg_dir / f"{model_yaml_name}.yaml"
        if candidate.exists():
            return OmegaConf.load(candidate)
    raise FileNotFoundError(f"Model YAML not found: {model_yaml_name}")


def list_dataset_model_pairs(cfg: Any) -> dict[str, dict[str, Any]]:
    data_cfg = cfg.data
    config_dirs = list(data_cfg.get("dataset_config_dirs", ["conf/data/datasets"]))
    pairs: dict[str, dict[str, Any]] = {}

    for cfg_dir in resolve_dirs(config_dirs):
        if not cfg_dir.exists():
            continue
        for path in sorted(cfg_dir.glob("*.yaml")):
            payload = OmegaConf.to_container(OmegaConf.load(path), resolve=True)
            if not isinstance(payload, dict):
                continue
            name = str(payload.get("name") or path.stem)
            pairs[name] = {
                "enabled": bool(payload.get("enabled", True)),
                "models": [str(item) for item in payload.get("models", [])],
            }

    return dict(sorted(pairs.items()))


def validate_selection(dataset_cfg_dict: dict[str, Any], model_yaml_name: str) -> None:
    compatible_models = [str(item) for item in dataset_cfg_dict.get("models", [])]
    if model_yaml_name not in compatible_models:
        raise ValueError(
            f"Model '{model_yaml_name}' is not compatible with dataset '{dataset_cfg_dict.get('name')}'. "
            f"Allowed models: {compatible_models}"
        )


def ensure_gated_access(cfg: Any, dataset_cfg_dict: dict[str, Any], model_cfg_dict: dict[str, Any]) -> None:
    needs_token = bool(dataset_cfg_dict.get("gated", False)) or bool(model_cfg_dict.get("gated", False))
    if needs_token and not get_hf_token(cfg):
        raise ValueError(
            "Selected dataset/model requires HF token. Set a valid value in conf/config.yaml -> hf.token"
        )


def show_image_grid(images: list[Any], max_images: int = 12, title: str | None = None) -> None:
    if not images:
        print("Empty image list")
        return

    picked = images[:max_images]
    cols = min(4, len(picked))
    rows = math.ceil(len(picked) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_array = np.array(axes).reshape(rows, cols)

    idx = 0
    for r in range(rows):
        for c in range(cols):
            ax = axes_array[r, c]
            ax.axis("off")
            if idx < len(picked):
                ax.imshow(picked[idx])
                ax.set_title(f"#{idx}")
            idx += 1

    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def fetch_batch_with_retries(raw_ds: Any, batch_size: int, attempts: int = 10, sleep_s: float = 1.0):
    for attempt in range(1, attempts + 1):
        batch = raw_ds.get_batch(batch_size)
        if batch:
            return batch
        print(f"Attempt {attempt}/{attempts}: empty batch, waiting {sleep_s:.1f}s...")
        time.sleep(sleep_s)
    return []


def pick_tensor(record: Any, target_kind: str) -> torch.Tensor:
    if target_kind == "inputs":
        return record.inputs
    if target_kind == "outputs":
        return record.outputs
    raise ValueError("target_kind must be 'inputs' or 'outputs'")


def sample_flat_values(tensor: torch.Tensor, max_points: int, seed: int) -> torch.Tensor:
    flat = tensor.detach().to(device="cpu", dtype=torch.float32).reshape(-1)
    if flat.numel() <= max_points:
        return flat
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)
    idx = torch.randperm(flat.numel(), generator=generator)[:max_points]
    return flat[idx]


def select_top_layers(layer_records: list[Any], target_kind: str, top_k: int) -> list[Any]:
    ordered = sorted(
        layer_records,
        key=lambda rec: int(pick_tensor(rec, target_kind).shape[0]),
        reverse=True,
    )
    return ordered[:top_k]


def build_layer_summary(
    layer_records: list[Any],
    target_kind: str,
    max_points: int,
    seed: int,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    quantiles = torch.tensor([0.01, 0.5, 0.99], dtype=torch.float32)

    for idx, rec in enumerate(layer_records):
        tensor = pick_tensor(rec, target_kind)
        n_rows = int(tensor.shape[0]) if tensor.ndim > 0 else 0
        dim = int(tensor.shape[1]) if tensor.ndim == 2 else 1

        if tensor.numel() == 0:
            rows.append(
                {
                    "layer": rec.layer_name,
                    "rows": n_rows,
                    "dim": dim,
                    "mean": float("nan"),
                    "std": float("nan"),
                    "p01": float("nan"),
                    "p50": float("nan"),
                    "p99": float("nan"),
                }
            )
            continue

        values = sample_flat_values(tensor, max_points=max_points, seed=seed + idx)
        q = torch.quantile(values, quantiles)
        rows.append(
            {
                "layer": rec.layer_name,
                "rows": n_rows,
                "dim": dim,
                "mean": float(values.mean()),
                "std": float(values.std(unbiased=False)),
                "p01": float(q[0]),
                "p50": float(q[1]),
                "p99": float(q[2]),
            }
        )

    return rows


def print_layer_summary(rows: list[dict[str, Any]]) -> None:
    if not rows:
        print("No rows to print")
        return

    header = (
        f"{'layer':56} {'rows':>8} {'dim':>6} "
        f"{'mean':>10} {'std':>10} {'p01':>10} {'p50':>10} {'p99':>10}"
    )
    print(header)
    print("-" * len(header))
    for row in rows:
        layer = (row['layer'][:53] + "...") if len(row['layer']) > 56 else row['layer']
        print(
            f"{layer:56} {row['rows']:8d} {row['dim']:6d} "
            f"{row['mean']:10.4f} {row['std']:10.4f} {row['p01']:10.4f} "
            f"{row['p50']:10.4f} {row['p99']:10.4f}"
        )


def plot_layer_histograms(
    layer_records: list[Any],
    target_kind: str,
    bins: int,
    max_points: int,
    seed: int,
) -> None:
    if not layer_records:
        print("No layers for histogram")
        return

    cols = 2 if len(layer_records) > 1 else 1
    rows = math.ceil(len(layer_records) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 3.8 * rows))
    axes_array = np.array(axes).reshape(rows, cols)

    for idx, rec in enumerate(layer_records):
        ax = axes_array[idx // cols, idx % cols]
        tensor = pick_tensor(rec, target_kind)
        values = sample_flat_values(tensor, max_points=max_points, seed=seed + idx).numpy()
        ax.hist(values, bins=bins, color="#2a9d8f", alpha=0.85)
        dim = int(tensor.shape[1]) if tensor.ndim == 2 else 1
        ax.set_title(f"{rec.layer_name}\nrows={tensor.shape[0]}, dim={dim}")
        ax.set_xlabel("activation value")
        ax.set_ylabel("count")

    for idx in range(len(layer_records), rows * cols):
        axes_array[idx // cols, idx % cols].axis("off")

    fig.suptitle(f"Layer-wise activation histograms ({target_kind})")
    plt.tight_layout()
    plt.show()


def plot_quantile_heatmap(
    layer_records: list[Any],
    target_kind: str,
    max_points: int,
    seed: int,
) -> None:
    if not layer_records:
        print("No layers for quantile heatmap")
        return

    q_levels = torch.tensor([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99], dtype=torch.float32)
    matrix: list[np.ndarray] = []
    labels: list[str] = []

    for idx, rec in enumerate(layer_records):
        tensor = pick_tensor(rec, target_kind)
        if tensor.numel() == 0:
            continue
        values = sample_flat_values(tensor, max_points=max_points, seed=seed + idx)
        q_values = torch.quantile(values, q_levels).numpy()
        matrix.append(q_values)
        labels.append(rec.layer_name)

    if not matrix:
        print("No non-empty layers for heatmap")
        return

    heat = np.vstack(matrix)
    fig_h = max(3.0, 0.55 * len(labels) + 1.6)
    fig, ax = plt.subplots(figsize=(10, fig_h))
    im = ax.imshow(heat, aspect="auto", cmap="coolwarm")

    ax.set_xticks(np.arange(len(q_levels)))
    ax.set_xticklabels([f"{float(q):.2f}" for q in q_levels])
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels([(name[:64] + "...") if len(name) > 67 else name for name in labels])
    ax.set_xlabel("quantile")
    ax.set_ylabel("layer")
    ax.set_title(f"Quantile heatmap by layer ({target_kind})")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("activation value")
    plt.tight_layout()
    plt.show()


In [ ]:
cfg_preview = load_hydra_cfg(overrides=[f"data.path={DATA_ROOT}"])
pairs = list_dataset_model_pairs(cfg_preview)

print("Available dataset -> models from YAML configs:")
for dataset_name, info in pairs.items():
    status = "enabled" if info["enabled"] else "disabled"
    models = ", ".join(info["models"]) if info["models"] else "<empty>"
    print(f"- {dataset_name} ({status}): {models}")

if DATASET_YAML_NAME not in pairs:
    raise ValueError(f"Unknown dataset '{DATASET_YAML_NAME}'. Pick one from the list above.")

if MODEL_YAML_NAME not in pairs[DATASET_YAML_NAME]["models"]:
    raise ValueError(
        f"Pair is not compatible: {DATASET_YAML_NAME} -> {MODEL_YAML_NAME}. "
        f"Allowed: {pairs[DATASET_YAML_NAME]['models']}"
    )

if TARGET_KIND not in {"inputs", "outputs"}:
    raise ValueError("TARGET_KIND must be 'inputs' or 'outputs'")

print("Selected pair is compatible.")


In [ ]:
cfg = load_hydra_cfg(
    overrides=[
        f"data.path={DATA_ROOT}",
        f"data.enabled_datasets=[{DATASET_YAML_NAME}]",
    ]
)

dataset_cfg = resolve_dataset_cfg(cfg, DATASET_YAML_NAME)
model_cfg = resolve_model_cfg(cfg, MODEL_YAML_NAME)

dataset_cfg_dict = OmegaConf.to_container(dataset_cfg, resolve=True)
model_cfg_dict = OmegaConf.to_container(model_cfg, resolve=True)
global_cfg_dict = OmegaConf.to_container(cfg, resolve=True)

validate_selection(dataset_cfg_dict, MODEL_YAML_NAME)
ensure_gated_access(cfg, dataset_cfg_dict, model_cfg_dict)

print("Dataset:", dataset_cfg_dict["name"])
print("Dataset gated:", dataset_cfg_dict.get("gated", False))
print("Model:", model_cfg_dict["name"])
print("Model gated:", model_cfg_dict.get("gated", False))
print("Model run_mode:", model_cfg_dict.get("run_mode"))
print("Model hook_filter:", model_cfg_dict.get("hook_filter"))


In [ ]:
if raw_dataset is not None:
    try:
        raw_dataset.close()
    except Exception:
        pass
    raw_dataset = None

register_all_adapters()
raw_dataset = create_dataset(
    name=DATASET_YAML_NAME,
    cfg=dataset_cfg_dict,
    global_root=str(DATA_ROOT),
    seed=int(cfg.data.seed),
    hf_cfg=OmegaConf.to_container(cfg.hf, resolve=True),
)
raw_dataset.start()

samples = fetch_batch_with_retries(raw_dataset, batch_size=N_IMAGES, attempts=12, sleep_s=1.0)
pil_batch = [item.image for item in samples]

print(f"Collected {len(pil_batch)} images from dataset '{DATASET_YAML_NAME}'")
if not pil_batch:
    raise RuntimeError("Failed to fetch non-empty batch from dataset")

show_image_grid(
    pil_batch,
    max_images=min(len(pil_batch), 12),
    title=f"Dataset batch: {DATASET_YAML_NAME}",
)

print("raw_dataset.stats():")
print(raw_dataset.stats())


In [ ]:
if model is not None:
    try:
        model.unload()
    except Exception:
        pass
    model = None

model_cfg_runtime = dict(model_cfg_dict)
model_cfg_runtime["device"] = DEVICE_OVERRIDE

model = create_model(MODEL_YAML_NAME, model_cfg_runtime, global_cfg_dict)
model.load()

layer_records = model.run(pil_batch)
print("LayerIORecord count:", len(layer_records))
if not layer_records:
    raise RuntimeError("No LayerIORecord produced")

print("model.stats():")
print(model.stats())


In [ ]:
selected_records = select_top_layers(layer_records, target_kind=TARGET_KIND, top_k=TOP_K_LAYERS)
summary_rows = build_layer_summary(
    selected_records,
    target_kind=TARGET_KIND,
    max_points=MAX_POINTS_PER_LAYER,
    seed=RANDOM_SEED,
)

print(f"Top {len(selected_records)} layers by number of rows ({TARGET_KIND}):")
print_layer_summary(summary_rows)


In [ ]:
plot_layer_histograms(
    selected_records,
    target_kind=TARGET_KIND,
    bins=HIST_BINS,
    max_points=MAX_POINTS_PER_LAYER,
    seed=RANDOM_SEED,
)


In [ ]:
plot_quantile_heatmap(
    selected_records,
    target_kind=TARGET_KIND,
    max_points=MAX_POINTS_PER_LAYER,
    seed=RANDOM_SEED,
)


Для сравнения `inputs` и `outputs` просто поменяйте `TARGET_KIND` в первой ячейке и перезапустите ноутбук.


In [ ]:
for obj_name in ["model", "raw_dataset"]:
    obj = globals().get(obj_name)
    if obj is None:
        continue
    try:
        if obj_name == "model":
            obj.unload()
        else:
            obj.close()
    except Exception as exc:
        print(f"Cleanup warning for {obj_name}: {exc}")

model = None
raw_dataset = None
print("Cleanup complete")
